In [69]:
import importlib.util
import subprocess
import sys
import ast

def is_package_installed(package_name):
    spec = importlib.util.find_spec(package_name)
    return spec is not None

def install_package(package_name):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

packages = ['keras', 'numpy', 'pandas', 'scikit-learn', 'torch', 'transformers']

for package in packages:
    if not is_package_installed(package):
        try:
            print(f"Installing {package}...")
            install_package(package)
            print(f"Installed {package}")
        except subprocess.CalledProcessError as error:
            print(f"Error installing {package}: {error}")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from transformers import AdamW, BertConfig, BertForSequenceClassification, BertModel, BertTokenizer
from tqdm import trange

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# device = torch.device("cpu")  # Temporarily run on CPU
# model.to(device)  # Move model to CPU first for debugging

Installing scikit-learn...
Installed scikit-learn


Using raw natural sentences for transformer models would have been better, but our assignment requires us to use data cleaned by ourself (to validate the cleaning was correct).

In [70]:
df = pd.read_csv("/kaggle/input/nlp-clean/BERT_data_final.csv", index_col=0)
df.head()

,category,resume
0,Hadoop,technical skill set program language APACHE HA...
1,Hadoop,service description project description data w...
2,Hadoop,order increase performance minimize HADOOP res...
3,Hadoop,objective project speed data processing analys...
4,Testing,good logical analytical skill positive attitud...


In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 444 entries, 0 to 443
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  444 non-null    object
 1   resume    444 non-null    object
dtypes: object(2)
memory usage: 10.4+ KB


In [72]:
data_frame = pd.DataFrame()
data_frame['sentence'] = df['resume']
data_frame['label'] = df['category']
data_frame.head()

,sentence,label
0,technical skill set program language APACHE HA...,Hadoop
1,service description project description data w...,Hadoop
2,order increase performance minimize HADOOP res...,Hadoop
3,objective project speed data processing analys...,Hadoop
4,good logical analytical skill positive attitud...,Testing


In [73]:
# Prepare features and labels for training and validation 
sentences = data_frame.sentence.values
sentences = ["[CLS] " + sentence + " [SEP]" for sentence in sentences]
labels = data_frame.label.values

In [74]:
# Generate attention masks for training and validation  
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
tokenized_sentences = [tokenizer.tokenize(sentence) for sentence in sentences]
input_id_sequences = [tokenizer.convert_tokens_to_ids(tokenized_sentence) for tokenized_sentence in tokenized_sentences]
input_id_sequences = pad_sequences(input_id_sequences, maxlen=256, dtype="long", truncating="post", padding="post")
attention_masks = [[float(i > 0) for i in input_id_sequence] for input_id_sequence in input_id_sequences]

In [75]:
# Prepare datasets for training and validation 
training_dataset, validation_dataset, training_labels, validation_labels, training_masks, validation_masks = train_test_split(input_id_sequences,
labels, attention_masks, random_state=2018, test_size=0.1)

# Initialize the encoder
label_encoder = LabelEncoder()

# Fit and transform labels into numeric values
training_labels = label_encoder.fit_transform(training_labels)
validation_labels = label_encoder.transform(validation_labels) 

# Ensure labels and masks are in proper NumPy formats
training_labels = np.array(training_labels, dtype=np.int64)  # Ensure int format
validation_labels = np.array(validation_labels, dtype=np.int64)

training_masks = np.array(training_masks, dtype=np.int64)  # Ensure int format
validation_masks = np.array(validation_masks, dtype=np.int64)

# Ensure input IDs are in correct format
training_dataset = np.array(training_dataset, dtype=np.int64)
validation_dataset = np.array(validation_dataset, dtype=np.int64)

batch_size = 16

training_dataset = TensorDataset(torch.tensor(training_dataset).to(device), torch.tensor(training_masks).to(device), torch.tensor(training_labels, dtype=torch.long).to(device))
training_sampler = RandomSampler(training_dataset)
training_dataloader = DataLoader(training_dataset, sampler=training_sampler, batch_size=batch_size)

validation_dataset = TensorDataset(torch.tensor(validation_dataset).to(device), torch.tensor(validation_masks).to(device), torch.tensor(validation_labels, dtype=torch.long).to(device))
validation_sampler = SequentialSampler(validation_dataset)
validation_dataloader = DataLoader(validation_dataset, sampler=validation_sampler, batch_size=batch_size)

num_classes = len(np.unique(training_labels))

# Raw bert-base-uncased from BrightSpace

In [76]:
# Configure model
configuration = BertConfig()
model = BertModel(configuration)
config = model.config

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_classes)
model = nn.DataParallel(model)
model.to(device)

parameters = list(model.named_parameters())
no_decay = ['bias', 'LayerNorm.weight']
parameters = [
    {'params': [p for n, p in parameters if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in parameters if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
optimizer = AdamW(parameters, lr=2e-5, correct_bias=False)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [77]:
# Calculate accuracy
def accuracy(predicted_labels, labels):
    predicted_labels = np.argmax(predicted_labels.to('cpu').numpy(), axis=1).flatten()
    labels = labels.to('cpu').numpy().flatten()
    return np.sum(predicted_labels == labels) / len(labels)

In [78]:
import time

# Train model
total_epochs = 20
training_losses = []

start = time.time()
for epoch in trange(total_epochs, desc="Epoch"):
    model.train()
    training_loss = 0
    training_steps = 0
    training_correct = 0  
    training_total = 0  
    
    for step, batch in enumerate(training_dataloader):
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        outputs = model(inputs, attention_mask=attention_masks, labels=labels)
        loss = outputs.loss
        logits = outputs.logits
        
        loss.backward()
        optimizer.step()

        training_loss += loss.item()
        training_steps += 1

        training_losses.append(loss.item())
        
        preds = torch.argmax(logits, dim=1)  
        training_correct += (preds == labels).sum().item()
        training_total += labels.size(0)

    average_training_loss = training_loss/training_steps
    training_accuracy = training_correct / training_total  

    print("Epoch {}: Average Training Loss: {:.4f}".format(epoch+1, average_training_loss))
    print("Epoch {}: Training Accuracy: {:.4f}".format(epoch+1, training_accuracy))

    model.eval()
    validation_accuracy = 0
    validation_steps = 0

    for batch in validation_dataloader:
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        with torch.no_grad():
            outputs = model(inputs, attention_mask=attention_masks, labels=labels)

        logits = outputs.logits
        temp_validation_accuracy = accuracy(logits, labels)
        validation_accuracy += temp_validation_accuracy
        validation_steps += 1

    average_validation_accuracy = validation_accuracy/validation_steps
    print("Epoch {}: Validation Accuracy: {:.4f}".format(epoch+1, average_validation_accuracy))

raw_duration = time.time() - start

Epoch:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1: Average Training Loss: 2.8779
Epoch 1: Training Accuracy: 0.2055


Epoch:   5%|▌         | 1/20 [00:09<03:08,  9.90s/it]

Epoch 1: Validation Accuracy: 0.3413
Epoch 2: Average Training Loss: 2.2808
Epoch 2: Training Accuracy: 0.3734


Epoch:  10%|█         | 2/20 [00:19<02:57,  9.88s/it]

...

Epoch 18: Validation Accuracy: 0.8397
Epoch 19: Average Training Loss: 0.0356
Epoch 19: Training Accuracy: 1.0000


Epoch:  95%|█████████▌| 19/20 [03:10<00:10, 10.12s/it]

Epoch 19: Validation Accuracy: 0.8397
Epoch 20: Average Training Loss: 0.0338
Epoch 20: Training Accuracy: 1.0000


Epoch: 100%|██████████| 20/20 [03:20<00:00, 10.03s/it]

Epoch 20: Validation Accuracy: 0.8397


In [79]:
# the model works and has 'acceptable' validation accuracy but is obviously overfitted

# Fine-tuning BERT 

In [80]:
from torch.optim.lr_scheduler import CosineAnnealingLR

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_classes,
    hidden_dropout_prob=0.3,  # Increased to 0.3 for better regularization
    attention_probs_dropout_prob=0.3  # Prevent overfitting in attention layers
)

model = nn.DataParallel(model)
model.to(device)

parameters = list(model.named_parameters())
no_decay = ['bias', 'LayerNorm.weight']
parameters = [
    {'params': [p for n, p in parameters if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in parameters if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]

optimizer = AdamW(parameters, lr=2e-5, correct_bias=False)

# Add a scheduler for learning rate decay
lr_scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs, eta_min=1e-6)

# Freeze Lower BERT Layers
for param in model.module.bert.embeddings.parameters():
    param.requires_grad = False
for layer in model.module.bert.encoder.layer[:4]:  # Freeze first 8 layers
    for param in layer.parameters():
        param.requires_grad = False

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [81]:
import torch.nn.utils as nn_utils

# Train model
total_epochs = 100 # Early-stopping will stop the training if necessary
training_losses = []

# Early-stopping variables
best_val_acc = 0
patience = 7  # Stop training if no improvement for 7 consecutive epochs
counter = 0

# Path to save the best model
best_model_path = "/kaggle/working/best_bert_model.pth"

start = time.time()
for epoch in trange(total_epochs, desc="Epoch"):
    model.train()
    training_loss = 0
    training_steps = 0
    training_correct = 0  
    training_total = 0  

    if epoch == 10:
        for param in model.module.bert.parameters():
            param.requires_grad = True
        print("Unfreezing all BERT layers")
    
    for step, batch in enumerate(training_dataloader):
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        outputs = model(inputs, attention_mask=attention_masks, labels=labels)
        loss = outputs.loss
        logits = outputs.logits
        
        loss.backward()
        
        # Apply Gradient Clipping (New)
        nn_utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
        optimizer.step()

        # Update learning rate scheduler
        lr_scheduler.step()  

        training_loss += loss.item()
        training_steps += 1

        training_losses.append(loss.item())
        
        preds = torch.argmax(logits, dim=1)  
        training_correct += (preds == labels).sum().item()
        training_total += labels.size(0)

    average_training_loss = training_loss/training_steps
    training_accuracy = training_correct / training_total  

    print("Epoch {}: Average Training Loss: {:.4f}".format(epoch+1, average_training_loss))
    print("Epoch {}: Training Accuracy: {:.4f}".format(epoch+1, training_accuracy))

    model.eval()
    validation_accuracy = 0
    validation_steps = 0

    for batch in validation_dataloader:
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        with torch.no_grad():
            outputs = model(inputs, attention_mask=attention_masks, labels=labels)

        logits = outputs.logits
        temp_validation_accuracy = accuracy(logits, labels)
        validation_accuracy += temp_validation_accuracy
        validation_steps += 1

    average_validation_accuracy = validation_accuracy/validation_steps
    print("Epoch {}: Validation Accuracy: {:.4f}".format(epoch+1, average_validation_accuracy))

    # Save best model
    if average_validation_accuracy > best_val_acc:
        best_val_acc = average_validation_accuracy
        counter = 0  # Reset patience counter
        
        # Save the model
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model saved at epoch {epoch+1} with validation accuracy {best_val_acc:.4f}")

    else:
        counter += 1  # Increment counter if no improvement
        print(f"No improvement, patience count: {counter}/{patience}")
        
    # Early Stopping Check    
    if counter >= patience:
        print("Early stopping triggered. Training stopped.")
        break  # Stop training if no improvement for n consecutive epochs
duration = time.time() - start

Epoch:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1: Average Training Loss: 3.1490
Epoch 1: Training Accuracy: 0.1404
Epoch 1: Validation Accuracy: 0.2324


Epoch:   1%|          | 1/100 [00:08<14:34,  8.84s/it]

Best model saved at epoch 1 with validation accuracy 0.2324
Epoch 2: Average Training Loss: 2.8902
Epoch 2: Training Accuracy: 0.1980
Epoch 2: Validation Accuracy: 0.3205


Epoch:   2%|▏         | 2/100 [00:17<14:23,  8.81s/it]

...

Epoch 32: Validation Accuracy: 0.8606
No improvement, patience count: 5/7
Epoch 33: Average Training Loss: 0.0898
Epoch 33: Training Accuracy: 1.0000


Epoch:  33%|███▎      | 33/100 [05:27<11:25, 10.23s/it]

Epoch 33: Validation Accuracy: 0.8606
No improvement, patience count: 6/7
Epoch 34: Average Training Loss: 0.0774
Epoch 34: Training Accuracy: 1.0000


Epoch:  33%|███▎      | 33/100 [05:37<11:25, 10.23s/it]

Epoch 34: Validation Accuracy: 0.8862
No improvement, patience count: 7/7
Early stopping triggered. Training stopped.


In [82]:
# Reload the best saved model
model.load_state_dict(torch.load(best_model_path))
model.to(device)
model.eval()
print("Best model loaded successfully!")

<ipython-input-82-b3befedb24d7>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))


Best model loaded successfully!


In [83]:
# Prepare features and labels for evaluation 
df_test = pd.read_csv("/kaggle/input/nlp-clean/LIM JIN BIN_dataset.csv")

# Prepare features and labels for evaluation
data_frame = pd.DataFrame()
data_frame['sentence'] = df_test['Resume']
data_frame['label'] = df_test['Category']

# Convert sentences to BERT input format
sentences = data_frame.sentence.values
sentences = ["[CLS] " + sentence + " [SEP]" for sentence in sentences]

# Tokenization & Convert to Input IDs
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
tokenized_sentences = [tokenizer.tokenize(sentence) for sentence in sentences]
input_id_sequences = [tokenizer.convert_tokens_to_ids(sentence) for sentence in tokenized_sentences]
input_id_sequences = pad_sequences(input_id_sequences, maxlen=256, dtype="long", truncating="post", padding="post")

# Generate attention masks
attention_masks = [[float(i > 0) for i in seq] for seq in input_id_sequences]

# Encode Labels Using the Same LabelEncoder from Training
labels = data_frame.label.values
labels = label_encoder.transform(labels)  # Ensure label encoding matches training

# Convert Data to Torch Tensors
input_id_sequences = torch.tensor(input_id_sequences, dtype=torch.long)
attention_masks = torch.tensor(attention_masks, dtype=torch.long)
labels = torch.tensor(labels, dtype=torch.long)

# Create Prediction DataLoader
prediction_dataset = TensorDataset(input_id_sequences, attention_masks, labels)
prediction_sampler = SequentialSampler(prediction_dataset)  # Match training validation behavior
prediction_dataloader = DataLoader(prediction_dataset, sampler=prediction_sampler, batch_size=batch_size)

In [84]:
# Tracking variables
total_loss = 0
correct_predictions = 0
total_samples = 0
logits_set = []
labels_set = []
matthews_set = []

# Disable gradient calculations for evaluation
with torch.no_grad():
    for batch in prediction_dataloader:
        batch_input_ids, batch_attention_masks, batch_labels = batch
        batch_input_ids, batch_attention_masks, batch_labels = (
            batch_input_ids.to(device), batch_attention_masks.to(device), batch_labels.to(device)
        )

        # Forward pass
        outputs = model(batch_input_ids, attention_mask=batch_attention_masks, labels=batch_labels)
        loss = outputs.loss
        logits = outputs.logits

        # Track total loss
        total_loss += loss.item()

        # Convert logits to predictions
        preds = torch.argmax(logits, dim=1)

        # Track accuracy
        correct_predictions += (preds == batch_labels).sum().item()
        total_samples += batch_labels.size(0)

        # Store for MCC calculation
        logits_set.append(logits.cpu().numpy())
        labels_set.append(batch_labels.cpu().numpy())

# Compute per-batch MCC
for i in range(len(labels_set)):
    batch_mcc = matthews_corrcoef(labels_set[i], np.argmax(logits_set[i], axis=1).flatten())
    matthews_set.append(batch_mcc)
    print(f"Batch {i + 1}: MCC = {batch_mcc:.4f}")

# Compute overall evaluation metrics
overall_loss = total_loss / len(prediction_dataloader)
overall_accuracy = correct_predictions / total_samples
overall_mcc = np.mean(matthews_set)

print(f"\n Overall Accuracy: {overall_accuracy:.4f}")
print(f" Overall Loss: {overall_loss:.4f}")
print(f" Overall MCC: {overall_mcc:.4f}")

Batch 1: MCC = 0.8062
Batch 2: MCC = 0.9364
Batch 3: MCC = 0.7281
Batch 4: MCC = 0.8761
Batch 5: MCC = 0.7598
Batch 6: MCC = 0.9359
Batch 7: MCC = 0.8035
Batch 8: MCC = 0.7473
Batch 9: MCC = 0.8096
Batch 10: MCC = 0.8718
Batch 11: MCC = 0.8686
Batch 12: MCC = 0.9342
Batch 13: MCC = 0.8077
Batch 14: MCC = 0.7544
Batch 15: MCC = 1.0000
Batch 16: MCC = 0.8713
Batch 17: MCC = 0.7401
Batch 18: MCC = 0.8175
Batch 19: MCC = 0.8077
Batch 20: MCC = 0.7403
Batch 21: MCC = 0.7517
Batch 22: MCC = 0.8106
Batch 23: MCC = 0.9354
Batch 24: MCC = 0.7457
Batch 25: MCC = 0.9356
Batch 26: MCC = 0.8000
Batch 27: MCC = 0.9356
Batch 28: MCC = 0.9353
Batch 29: MCC = 0.8690
Batch 30: MCC = 0.9345
Batch 31: MCC = 0.8079
Batch 32: MCC = 0.8095
Batch 33: MCC = 0.8771
Batch 34: MCC = 0.9348
Batch 35: MCC = 0.8062
Batch 36: MCC = 0.8764
Batch 37: MCC = 0.8793
Batch 38: MCC = 1.0000
Batch 39: MCC = 0.8139
Batch 40: MCC = 1.0000
Batch 41: MCC = 0.9359
Batch 42: MCC = 0.9348
Batch 43: MCC = 0.8060
Batch 44: MCC = 0.87

In [85]:
print("There might be supposedly overlapping instances between my training data and Jin Bin's BrightSpace dataset, but currently, this is the only test dataset I can get and a MCC overall score of 0.815 and test accuracy of 0.817 indicates that my model is well-trained and performs well.")

There might be supposedly overlapping instances between my training data and Jin Bin's BrightSpace dataset, but currently, this is the only test dataset I can get and a MCC overall score of 0.815 and test accuracy of 0.817 indicates that my model is well-trained and performs well.


In [86]:
# Track total loss and correct predictions
total_loss = 0
correct_predictions = 0
total_samples = 0

# Disable gradient calculations for evaluation
with torch.no_grad():
    for batch in validation_dataloader:
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        # Forward pass
        outputs = model(inputs, attention_mask=attention_masks, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        # Calculate loss
        total_loss += loss.item()

        # Get predictions
        preds = torch.argmax(logits, dim=1)

        # Update accuracy calculations
        correct_predictions += (preds == labels).sum().item()
        total_samples += labels.size(0)

# Compute average loss and accuracy
average_loss = total_loss / len(prediction_dataloader)
accuracy = correct_predictions / total_samples

print(f"Validatin Dataset Accuracy: {accuracy:.4f}")
print(f"Validation Dataset Loss: {average_loss:.4f}")

Validatin Dataset Accuracy: 0.8889
Validation Dataset Loss: 0.0245


In [87]:
# P.S. Val accuray if 0.933 for the best saved model (for marking purposes)

# Optimization

In [99]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
encoding = tokenizer(
    df["resume"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

train_dataloader = DataLoader(training_dataset, batch_size=16, num_workers=4, pin_memory=True)
val_dataloader = DataLoader(validation_dataset, batch_size=16, num_workers=4, pin_memory=True)

In [102]:
from torch.optim.lr_scheduler import CosineAnnealingLR

model_opti = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_classes,
    hidden_dropout_prob=0.4,  # Increased to 0.4 for better regularization
    attention_probs_dropout_prob=0.4  # Prevent overfitting in attention layers
)

model_opti = nn.DataParallel(model_opti)
model_opti.to(device)

parameters = list(model_opti.named_parameters())
no_decay = ['bias', 'LayerNorm.weight']
parameters = [
    {'params': [p for n, p in parameters if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in parameters if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]

optimizer = AdamW(parameters, lr=1e-5, correct_bias=False)

# Add a scheduler for learning rate decay
lr_scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs, eta_min=1e-6)

# Freeze Lower BERT Layers
for param in model_opti.module.bert.embeddings.parameters():
    param.requires_grad = False
for layer in model_opti.module.bert.encoder.layer[:4]:  # Freeze first 8 layers
    for param in layer.parameters():
        param.requires_grad = False

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [103]:
from torch.cuda.amp import autocast, GradScaler

# Enable mixed precision training
scaler = GradScaler()
gradient_accumulation_steps = 3

best_val_acc = 0
patience = 7
counter = 0
best_model_path_opti = "/kaggle/working/best_bert_model_opti.pth"

start = time.time()
for epoch in trange(100, desc="Epoch"):  
    model_opti.train()
    training_loss = 0
    training_steps = 0
    training_correct = 0  
    training_total = 0  
            
    if epoch == 10:  # Unfreeze all layers after 10 epochs
        for param in model_opti.module.bert.parameters():
            param.requires_grad = True
        print("Unfreezing all BERT layers for full fine-tuning")

    for step, batch in enumerate(training_dataloader):
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        
        with autocast():  
            outputs = model_opti(inputs, attention_mask=attention_masks, labels=labels)
            loss = outputs.loss / gradient_accumulation_steps  

        scaler.scale(loss).backward()
        
        if (step + 1) % gradient_accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        training_loss += loss.item()
        training_steps += 1

        preds = torch.argmax(outputs.logits, dim=1)  
        training_correct += (preds == labels).sum().item()
        training_total += labels.size(0)

    # Compute average training loss & accuracy
    average_training_loss = training_loss / training_steps
    training_accuracy = training_correct / training_total  

    print("Epoch {}: Average Training Loss: {:.4f}".format(epoch+1, average_training_loss))
    print("Epoch {}: Training Accuracy: {:.4f}".format(epoch+1, training_accuracy))

    # Validation Step
    model_opti.eval()
    validation_accuracy = 0
    validation_steps = 0

    for batch in validation_dataloader:
        inputs = batch[0].to(device)
        attention_masks = batch[1].to(device)
        labels = batch[2].to(device)

        with torch.no_grad():
            outputs = model_opti(inputs, attention_mask=attention_masks, labels=labels)

        logits = outputs.logits
        temp_validation_accuracy = (torch.argmax(logits, dim=1) == labels).sum().item() / labels.size(0)
        validation_accuracy += temp_validation_accuracy
        validation_steps += 1

    # Compute average validation accuracy
    average_validation_accuracy = validation_accuracy / validation_steps
    print("Epoch {}: Validation Accuracy: {:.4f}".format(epoch+1, average_validation_accuracy))

    # Save best model
    if average_validation_accuracy > best_val_acc:
        best_val_acc = average_validation_accuracy
        torch.save(model_opti.state_dict(), best_model_path_opti)
        print(f"Best model saved at epoch {epoch+1} with validation accuracy {best_val_acc:.4f}")
        counter = 0  # Reset patience counter
    else:
        counter += 1
        print(f"No improvement, patience count: {counter}/{patience}")

    # Early Stopping    
    if counter >= patience:
        print("Early stopping triggered. Training stopped.")
        break

# Track total training duration
opti_duration = time.time() - start

<ipython-input-103-af44f040922b>:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch:   0%|          | 0/100 [00:00<?, ?it/s]<ipython-input-103-af44f040922b>:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1: Average Training Loss: 1.0936
Epoch 1: Training Accuracy: 0.0351
Epoch 1: Validation Accuracy: 0.0256


Epoch:   1%|          | 1/100 [00:08<13:23,  8.12s/it]

Best model saved at epoch 1 with validation accuracy 0.0256
Epoch 2: Average Training Loss: 1.0424
Epoch 2: Training Accuracy: 0.1203
Epoch 2: Validation Accuracy: 0.2324


Epoch:   2%|▏         | 2/100 [00:16<13:12,  8.09s/it]

...

Epoch 27: Validation Accuracy: 0.3670
No improvement, patience count: 5/7
Epoch 28: Average Training Loss: 0.7137
Epoch 28: Training Accuracy: 0.3810


Epoch:  28%|██▊       | 28/100 [04:03<11:07,  9.26s/it]

Epoch 28: Validation Accuracy: 0.3622
No improvement, patience count: 6/7
Epoch 29: Average Training Loss: 0.6934
Epoch 29: Training Accuracy: 0.3835


Epoch:  28%|██▊       | 28/100 [04:13<10:50,  9.04s/it]

Epoch 29: Validation Accuracy: 0.3622
No improvement, patience count: 7/7
Early stopping triggered. Training stopped.


In [92]:
# The accuracy was sacrificed for faster training speed, but still accpetable. 
# Val Accuracy was slightly reduced from 0.9333 to 0.8606. 
# But, training time per epoch increased from 10s/it to 5s/it.

In [93]:
# Load the best saved optimized model
model_opti.load_state_dict(torch.load(best_model_path_opti))
model_opti.to(device)
model_opti.eval()
print("Optimized model loaded successfully!")

<ipython-input-93-de2dae7077cf>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_opti.load_state_dict(torch.load(best_model_path_opti))


Optimized model loaded successfully!


In [94]:
# Let us check the overall training time
print(f"BERT Fine-tuning total training time: {duration:.2f} seconds.")
print(f"BERT Fine-tuning avg training time per epoch: {duration/37:.2f} seconds.")

BERT Fine-tuning total training time: 337.66 seconds.
BERT Fine-tuning avg training time per epoch: 9.13 seconds.


In [95]:
print(f"BERT Optimization total training time: {opti_duration:.2f} seconds.")
print(f"BERT Optimization avg training time per epoch: {opti_duration/47:.2f} seconds.")

BERT Optimization total training time: 67.81 seconds.
BERT Optimization avg training time per epoch: 1.44 seconds.


In [96]:
# Let us check the inference time per each new prediction

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Example text input from ChatGPT aimed for a data scientist
# Prompt: give me a longer sample text like this aimed for a data scientist:
# text = "I have experience in machine learning and software engineering."
text = """
I am an experienced Data Scientist with a strong background in machine learning, deep learning, and statistical modeling.
I have worked extensively with Python, R, and SQL to build predictive models and data-driven solutions. My expertise includes 
natural language processing (NLP), computer vision, and time series forecasting. I have hands-on experience in deploying 
machine learning models using TensorFlow, PyTorch, and Scikit-learn. I have also worked with cloud platforms like AWS, GCP, 
and Azure to deploy scalable machine learning applications. My work involves data cleaning, feature engineering, and hyperparameter 
tuning to optimize model performance. I am proficient in big data technologies like Spark, Hadoop, and Apache Kafka, and I have 
experience in working with structured and unstructured data. I have successfully led data science projects in industries such as 
finance, healthcare, and e-commerce, improving decision-making and business intelligence. My strong communication skills allow me 
to present complex data insights to both technical and non-technical stakeholders, ensuring actionable results. 
"""
start = time.time()
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=256)

# Move inputs to the same device as the model
inputs = {key: value.to("cpu") for key, value in inputs.items()}

# Run inference
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Get predicted class
predicted_class = torch.argmax(logits, dim=1).item()
print(f"Predicted Category: {predicted_class}")
predicted_category = label_encoder.inverse_transform([predicted_class])[0]
print(f"Predicted Job Role: {predicted_category}")
fine_tune_inference = time.time() - start
print(f"Fine-tuned BERT took {fine_tune_inference:.6f} seconds.")

Predicted Category: 6
Predicted Job Role: Data Science
Fine-tuned BERT took 0.019913 seconds.


In [97]:
start = time.time()
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)

# Move inputs to the same device as the model
inputs = {key: value.to("cpu") for key, value in inputs.items()}

# Run inference
with torch.no_grad():
    outputs = model_opti(**inputs)
    logits = outputs.logits

# Get predicted class
predicted_class = torch.argmax(logits, dim=1).item()
print(f"Predicted Category: {predicted_class}")
predicted_category = label_encoder.inverse_transform([predicted_class])[0]
print(f"Predicted Job Role: {predicted_category}")
opti_inference = time.time() - start
print(f"Optimized BERT tooke {opti_inference:.6f} seconds.")

Predicted Category: 23
Predicted Job Role: Testing
Optimized BERT tooke 0.019142 seconds.
